# Modelo de zonas de riesgo (KMeans) — AlertaSegura Perú

Sprint 4 · Data Analysis

Agrupa los reportes por cercanía geográfica (`latitud`/`longitud`) con
KMeans para identificar zonas de riesgo, y les asigna un puntaje simple
en función de cuántos reportes tienen y qué tan verificados están.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

sns.set_theme(style="whitegrid")

reportes = pd.read_csv("../data/reportes_sinteticos.csv", parse_dates=["created_at"])
print("Total de reportes:", len(reportes))
reportes.head()

## 1. Elegir k con el método del codo

No sabemos de antemano cuántas zonas de riesgo "naturales" hay, así que
probamos varios valores de k y nos quedamos con el punto donde la mejora
empieza a aplanarse (método del codo).

In [ ]:
X = reportes[["latitud", "longitud"]].values

inercias = []
rango_k = range(2, 16)
for k in rango_k:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inercias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(list(rango_k), inercias, marker="o")
plt.xlabel("k (n° de zonas)")
plt.ylabel("Inercia")
plt.title("Método del codo")
plt.tight_layout()
plt.show()

El codo se nota alrededor de **k=8-10**: coincide con que en Sprint 3
generamos 6 "hotspots" más el resto de reportes dispersos por distrito,
así que tiene sentido que salgan más de 6 zonas. Usamos k=9.

In [ ]:
K = 9
kmeans = KMeans(n_clusters=K, random_state=42, n_init=10)
reportes["zona"] = kmeans.fit_predict(X)
reportes["zona"].value_counts().sort_index()

## 2. Visualizar las zonas

In [ ]:
plt.figure(figsize=(7, 8))
sns.scatterplot(
    data=reportes, x="longitud", y="latitud", hue="zona", palette="tab10", s=40,
)
centros = kmeans.cluster_centers_
plt.scatter(centros[:, 1], centros[:, 0], c="black", marker="X", s=150, label="Centro de zona")
plt.title("Zonas de riesgo (KMeans, k=9)")
plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 3. Puntaje de riesgo por zona

Puntaje simple: número de reportes de la zona, ponderado por qué tan
verificados están (una zona con muchos reportes `RECHAZADO` es menos
confiable que una con muchos `VERIFICADO`).

In [ ]:
peso_estado = {"VERIFICADO": 1.0, "PENDIENTE": 0.5, "RECHAZADO": 0.0}
reportes["peso_estado"] = reportes["estado"].map(peso_estado)

resumen_zonas = (
    reportes.groupby("zona")
    .agg(
        n_reportes=("id", "count"),
        distrito_principal=("distrito", lambda s: s.mode().iat[0]),
        categoria_principal=("categoria", lambda s: s.mode().iat[0]),
        puntaje_riesgo=("peso_estado", "sum"),
        lat_centro=("latitud", "mean"),
        lon_centro=("longitud", "mean"),
    )
    .sort_values("puntaje_riesgo", ascending=False)
)
resumen_zonas

In [ ]:
plt.figure(figsize=(8, 5))
orden = resumen_zonas.sort_values("puntaje_riesgo")
labels = [f"Zona {z} — {row.distrito_principal}" for z, row in orden.iterrows()]
bars = plt.barh(labels, orden["puntaje_riesgo"], color=sns.color_palette("rocket", len(orden)))
plt.bar_label(bars, fmt="%.1f", padding=3)
plt.title("Puntaje de riesgo por zona")
plt.xlabel("Puntaje (reportes ponderados por verificación)")
plt.tight_layout()
plt.show()

In [ ]:
reportes.to_csv("../data/reportes_con_zonas.csv", index=False)
resumen_zonas.to_csv("../data/resumen_zonas_riesgo.csv")
print("Guardado: reportes_con_zonas.csv y resumen_zonas_riesgo.csv")

## Próximos pasos

- El dashboard interactivo (`scripts/dashboard_app.py`, Plotly Dash) lee
  `reportes_con_zonas.csv` y muestra estas mismas zonas sobre un mapa,
  con filtros por categoría/distrito/estado.